# 02 · Preprocesamiento
Limpieza, integración y construcción del dataset final con **rezago temporal estricto (t−1)**.

**Unidad de análisis:** estudiante-año (`MRUN`, `AGNO`). **Objetivo en el año t:** reprobado o retirado.
**Predictores:** rendimiento del año t−1 y estructura de matrícula del año t (conocida al inicio del año).

**Salidas:** `data/processed/dataset_final`, `results/cobertura.csv`, `results/diagnostico_calidad.csv`,
`results/sit_fin_vs_oficial.csv`, `results/auditoria_fuga.json`.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / 'src'))
import numpy as np, pandas as pd
import proyecto as P
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
print('Raíz del proyecto:', P.RAIZ)

## 1. Diagnóstico de calidad por año (archivo de Rendimiento)
Para cada año: filas, estudiantes únicos, registros duplicados, notas igual a 0 y distribución de `SIT_FIN`.
Las cifras se comparan con las **tabulaciones oficiales del Centro de Estudios** (porcentajes, para que la
comparación valga también si se usa la muestra del 5 %).

In [ ]:
diag, sit_tabs, ceros_tabs = [], [], []
for a in P.ANIOS:
    df, info = P.cargar_rendimiento(a)
    _, dup = P.resolver_duplicados(df)
    diag.append({**info, **dup})

    vc = df['sit'].replace('', '(vacío)').value_counts()
    tab = pd.DataFrame({'codigo': vc.index, 'n': vc.values, 'pct': (vc / vc.sum() * 100).round(2).values})
    tab['categoria_oficial_supuesta'] = tab['codigo'].map(P.MAPA_SIT_OFICIAL).fillna('otro/sin_informacion')
    of = pd.Series(P.OFICIAL_SIT[a]); of = of / of.sum() * 100
    tab['pct_oficial'] = tab['categoria_oficial_supuesta'].map(of).round(2)
    tab['anio'] = a
    sit_tabs.append(tab)

    z = df[df['PROM_GRAL'] == 0]
    ceros_tabs.append(z['COD_ENSE2'].value_counts().rename_axis('COD_ENSE2').reset_index(name='n_notas_cero').assign(anio=a))
    print(f'--- {a}: {info["filas"]:,} filas ({info["archivo"]})')
    del df, z

diag = pd.DataFrame(diag)
display(diag.T)
diag.to_csv(P.DIR_RES / 'diagnostico_calidad.csv', index=False)

### 1.1 Códigos de `SIT_FIN` frente a las cifras oficiales
**Revisar esta tabla antes de seguir:** el objetivo se define con `CODIGOS_POSITIVOS` / `CODIGOS_NEGATIVOS`
de `src/proyecto.py`. Si los porcentajes por código no calzan con el oficial (`pct` vs `pct_oficial`),
hay que corregir el mapeo. Cualquier código fuera de P/R/Y queda **excluido** del objetivo (p. ej. traslado) y se cuenta.

In [ ]:
sit_todo = pd.concat(sit_tabs, ignore_index=True)
sit_todo.to_csv(P.DIR_RES / 'sit_fin_vs_oficial.csv', index=False)
display(sit_todo)
print('Positivos (fracaso):', sorted(P.CODIGOS_POSITIVOS), '| Negativos (promovido):', sorted(P.CODIGOS_NEGATIVOS))

### 1.2 Notas iguales a 0,0: ¿dónde están?
Un promedio 0,0 no es una nota. Se muestra en qué nivel de enseñanza (`COD_ENSE2`) se concentran.
Si se concentran en un nivel (p. ej. educación especial, que desde 2024 viene sin notas), el 0,0 es
**ausencia de dato** y no un indicio de retiro.

In [ ]:
ceros = pd.concat(ceros_tabs, ignore_index=True)
ceros['pct_dentro_del_anio'] = (ceros['n_notas_cero'] / ceros.groupby('anio')['n_notas_cero'].transform('sum') * 100).round(1)
ceros.to_csv(P.DIR_RES / 'ceros_por_nivel.csv', index=False)
display(ceros.sort_values(['anio', 'n_notas_cero'], ascending=[True, False]).groupby('anio').head(5))

## 2. Construcción de cohortes (integración + rezago t−1)
Para cada año t objetivo (2024 y 2025):
1. Rendimiento *t*: se resuelven duplicados (un registro por `MRUN`, preferencia por situación final válida) y se
   calcula el objetivo.
2. Matrícula *t*: un registro por `MRUN`; aporta `GEN_ALU`, `COD_ENSE`, `COD_ENSE2`.
3. Rendimiento *t−1*: aporta promedio, asistencia y situación del año anterior.
4. Cruce por **`MRUN`** (un registro por estudiante en cada tabla, verificado con `validate='one_to_one'`).
   Solo entran quienes tienen registro en t−1; **se cuantifica a quienes quedan fuera** (ingresantes al sistema).

Ninguna calificación ni asistencia del año *t* entra a los predictores.

In [ ]:
cohortes, infos, auditorias = [], [], []
for a in P.ANIOS_OBJETIVO:
    d, info, extras = P.construir_cohorte(a)
    cohortes.append(d); infos.append(info); auditorias.append(extras['auditoria_fuga'])
    print(f'Cohorte {a}: {len(d):,} estudiantes con rezago t-1')
    print('  Sin registro en t-1 por nivel (COD_ENSE2), primeros:', extras['sin_prev_por_nivel'].head(5).to_dict())

cob = pd.DataFrame([{k: v for k, v in i.items() if not isinstance(v, dict)} for i in infos])
cob.to_csv(P.DIR_RES / 'cobertura.csv', index=False)
P.guardar_json(infos, 'cobertura_detalle.json')
display(cob.T)

## 3. Revisión de fuga de información (data leakage)
Se demuestra con datos por qué el promedio del año *t* **no** puede ser predictor: se calcula su relación
con el objetivo del mismo año.

In [ ]:
from sklearn.metrics import roc_auc_score
aud = pd.concat(auditorias, ignore_index=True).dropna(subset=['prom_t'])
aud = aud.sample(min(300_000, len(aud)), random_state=P.SEMILLA)
fuga = {'n': int(len(aud)),
        'spearman_prom_t_vs_objetivo': float(aud['prom_t'].corr(aud['objetivo'], method='spearman')),
        'auc_prom_t_vs_objetivo': float(roc_auc_score(aud['objetivo'], aud['prom_t']))}
P.guardar_json(fuga, 'auditoria_fuga.json')
print(fuga)

## 4. Dataset final, verificaciones y exportación

In [ ]:
final = pd.concat(cohortes, ignore_index=True)

# --- verificaciones
assert not final.duplicated(['MRUN', 'AGNO']).any(), 'hay (MRUN, AGNO) duplicados'
assert final[P.OBJETIVO].isin([0, 1]).all()
assert not any(c in final.columns for c in ('PROM_GRAL', 'prom', 'SIT_FIN', 'ASISTENCIA')), 'columna contemporánea al objetivo'
assert (final['prom_notas_norm_prev'].dropna().between(1, 7)).all()

ruta = P.guardar_tabla(final, 'dataset_final')
print('Guardado en', ruta)
print('Dimensiones:', final.shape)
print('Tasa de reprobación/retiro (objetivo=1): %.2f%%' % (final[P.OBJETIVO].mean() * 100))
display(final.dtypes.to_frame('dtype').T)
display(final.head())
display(final.groupby('AGNO')[P.OBJETIVO].agg(['size', 'sum', 'mean']).rename(columns={'size': 'n', 'sum': 'positivos', 'mean': 'tasa'}))